# Anti-Aliasing testing for full UCR dataset
Using resamples from reproduction

In [ ]:
import time
import numpy as np
import pandas as pd
from numba import njit
from aeon.datasets import load_from_ts_file
from sklearn.preprocessing import LabelEncoder
import os
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import f1_score

from minirocket_aa import fit, transform

### Initialisation

In [ ]:
DATA_DIR = "../datasets"
SPLIT_DIR = "../splits"
RESULTS_DIR = "../results_improvements"
REPRODUCTION_CSV = "../minirocket_ucr109_results/minirocket_ucr109_results/minirocket_per_resample.csv"

RUN_NAME = "aa"

NUM_FEATURES = 10_000
RESAMPLES = range(30)
DATASETS = None              # used for quick testing, None to run for all datasets

CLASSIFIERS = ["ridge"]

VARIANTS = {
    "raw": [0.0],                  # original MiniRocket
    "aa_sf_0_1": [0.0, 1.0],       # raw view + smoothing factor of 1
}

### Loading the data

In [3]:
def load_dataset(name):
    X_train, y_train = load_from_ts_file(f"{DATA_DIR}/{name}/{name}_TRAIN.ts")
    X_test, y_test = load_from_ts_file(f"{DATA_DIR}/{name}/{name}_TEST.ts")

    X_all = np.concatenate([X_train, X_test])[:, 0, :].astype(np.float32)
    y_all = LabelEncoder().fit_transform(np.concatenate([y_train, y_test]))

    return X_all, y_all, len(X_train)



if DATASETS is None:
    dataset_names = sorted(f[:-len("_splits.npz")] for f in os.listdir(SPLIT_DIR) if f.endswith("_splits.npz"))
else:
    dataset_names = list(DATASETS)

start = time.perf_counter()

data = {}
for name in dataset_names:
    data[name] = load_dataset(name)

print(f"Loaded {len(data)} datasets, {sum(len(y_all) for _, y_all, _ in data.values()):,} series ")

Loaded 109 datasets, 172,051 series 


### Loading the resamples

In [4]:
splits = {}

for name in dataset_names:

    X_all, y_all, n_train_original = data[name]
    original_counts = np.bincount(y_all[:n_train_original])

    splits[name] = {}

    with np.load(f"{SPLIT_DIR}/{name}_splits.npz") as file:

        assert int(file["n_train_original"]) == n_train_original, name

        for r in RESAMPLES:
            train_index = file[f"train_{r:02d}"]
            test_index = file[f"test_{r:02d}"]

            assert np.array_equal(np.sort(np.concatenate([train_index, test_index])), np.arange(len(y_all))), (name, r)

            assert np.array_equal(np.bincount(y_all[train_index], minlength = len(original_counts)), original_counts), (name, r)

            splits[name][r] = (train_index, test_index)

    if 0 in splits[name]:
        assert np.array_equal(splits[name][0][0], np.arange(n_train_original)), name

print(f"Splits OK: {len(splits)} datasets x {len(RESAMPLES)} resamples")

Splits OK: 109 datasets x 30 resamples


### Feature extraction with new Anti-Aliasing features

In [5]:
@njit
def seed_numba(seed):
    np.random.seed(seed)


def fit_views(X_train, smooth_factors, seed):
    views = []
    for smooth_factor in smooth_factors:
        seed_numba(seed)
        views.append(fit(
            X_train,
            num_features = NUM_FEATURES // len(smooth_factors),
            smooth_factor = smooth_factor
        ))
    return views


def minirocket_features(X, views):
    return np.hstack([transform(X, parameters) for parameters in views])

name = dataset_names[0]
X_all, y_all, n_train_original = data[name]
X_example = X_all[:n_train_original]

for variant, smooth_factors in VARIANTS.items():
    views = fit_views(X_example, smooth_factors, seed = 0)
    F = minirocket_features(X_example, views)
    print(f"{name} | {variant}: {X_example.shape} series -> {F.shape} features")
    for smooth_factor, (dilations, _, _, windows, _) in zip(smooth_factors, views):
        print(f"    smooth_factor {smooth_factor}: smoothing window per dilation {windows}")

ACSF1 | raw: (100, 1460) series -> (100, 9996) features
    smooth_factor 0.0: smoothing window per dilation [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
ACSF1 | aa_sf_0_1: (100, 1460) series -> (100, 9912) features
    smooth_factor 0.0: smoothing window per dilation [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
    smooth_factor 1.0: smoothing window per dilation [  1   2   3   4   5   6   7   8  10  12  14  17  20  24  28  34  40  47
  56  66  78  93 110 130 154 182]


### Model training

In [ ]:
RIDGE_ALPHAS = np.logspace(-3, 3, 10)


def run_ridge(X_train, y_train, X_test, y_test, smooth_factors, seed):
    # fit: choose dilations and biases
    start = time.perf_counter()
    views = fit_views(X_train, smooth_factors, seed)
    fit_seconds = time.perf_counter() - start

    # transform: series -> features
    start = time.perf_counter()
    F_train = minirocket_features(X_train, views)
    F_test = minirocket_features(X_test, views)
    transform_seconds = time.perf_counter() - start

    # training: normalise, fit the classifier, score the test set
    start = time.perf_counter()
    feature_mean = F_train.mean(axis = 0)
    F_train -= feature_mean
    F_test -= feature_mean
    feature_scale = np.sqrt((F_train ** 2).sum(axis = 0))
    feature_scale[feature_scale == 0] = 1.0
    F_train /= feature_scale
    F_test /= feature_scale

    model = RidgeClassifierCV(alphas = RIDGE_ALPHAS)
    model.fit(F_train, y_train)
    predictions = model.predict(F_test)
    accuracy = (predictions == y_test).mean()
    f1 = f1_score(y_test, predictions, average = "macro", zero_division = 0)
    train_seconds = time.perf_counter() - start

    return {
        "accuracy": accuracy,
        "f1": f1,
        "fit_seconds": fit_seconds,
        "transform_seconds": transform_seconds,
        "train_seconds": train_seconds,
    }


PER_RESAMPLE_CSV = f"{RESULTS_DIR}/{RUN_NAME}_ucr109_per_resample.csv"

try:
    saved = pd.read_csv(PER_RESAMPLE_CSV)
    if "classifier" not in saved.columns:
        raise ValueError(f"{PER_RESAMPLE_CSV} is from an older version without a 'classifier' column. Move or delete it to start fresh.")
    rows = saved.to_dict("records")
    print(f"Continuing: {len(rows)} results already in {PER_RESAMPLE_CSV}")
except FileNotFoundError:
    rows = []

done = {(row["dataset"], row["resample"], row["variant"], row["classifier"]) for row in rows}

run_start = time.perf_counter()

for dataset_number, name in enumerate(dataset_names, start = 1):

    X_all, y_all, _ = data[name]
    num_classes = int(y_all.max()) + 1

    for r in RESAMPLES:

        train_index, test_index = splits[name][r]

        X_train, y_train = X_all[train_index], y_all[train_index]
        X_test, y_test = X_all[test_index], y_all[test_index]

        for variant, smooth_factors in VARIANTS.items():
            for classifier in CLASSIFIERS:

                if (name, r, variant, classifier) in done:
                    continue

                start = time.perf_counter()

                if classifier == "ridge":
                    outcome = run_ridge(X_train, y_train, X_test, y_test, smooth_factors, seed = r)
                else:
                    raise ValueError(f"Unknown classifier: {classifier}")

                rows.append({
                    "dataset": name,
                    "resample": r,
                    "variant": variant,
                    "classifier": classifier,
                    **outcome,
                    "total_seconds": time.perf_counter() - start,
                })

    pd.DataFrame(rows).to_csv(PER_RESAMPLE_CSV, index = False)

    means = " | ".join(
        f"{classifier} {variant} " + "{:.4f}".format(np.mean([
            row["accuracy"] for row in rows
            if row["dataset"] == name and row["variant"] == variant and row["classifier"] == classifier
        ]))
        for classifier in CLASSIFIERS
        for variant in VARIANTS
    )
    print(f"{dataset_number:3d}/{len(dataset_names)} {name:30} {means} | {(time.perf_counter() - run_start) / 60:6.1f} min")

print(f"\nFinished. Per-resample results: {PER_RESAMPLE_CSV}")

  1/109 ACSF1                          ridge raw 0.8237 | ridge aa_sf_0_1 0.8350 |    0.4 min
  2/109 Adiac                          ridge raw 0.8084 | ridge aa_sf_0_1 0.8063 |    0.8 min
  3/109 ArrowHead                      ridge raw 0.8800 | ridge aa_sf_0_1 0.8806 |    0.9 min
  4/109 BME                            ridge raw 0.9922 | ridge aa_sf_0_1 0.9927 |    1.0 min
  5/109 Beef                           ridge raw 0.7844 | ridge aa_sf_0_1 0.7722 |    1.1 min
  6/109 BeetleFly                      ridge raw 0.9033 | ridge aa_sf_0_1 0.9133 |    1.2 min
  7/109 BirdChicken                    ridge raw 0.8967 | ridge aa_sf_0_1 0.9000 |    1.3 min
  8/109 CBF                            ridge raw 0.9971 | ridge aa_sf_0_1 0.9984 |    1.5 min
  9/109 Car                            ridge raw 0.9200 | ridge aa_sf_0_1 0.9128 |    1.7 min
 10/109 Chinatown                      ridge raw 0.9655 | ridge aa_sf_0_1 0.9663 |    1.7 min
 11/109 ChlorineConcentration          ridge raw 0.7539 | ri

### Results Output

In [ ]:
SUMMARY_CSV = f"{RESULTS_DIR}/{RUN_NAME}_summary.csv"
BASELINE = "raw"

results = pd.read_csv(PER_RESAMPLE_CSV)

classifiers = results["classifier"].unique().tolist()
variants = results["variant"].unique().tolist()
others = [variant for variant in variants if variant != BASELINE]

results["run"] = results["classifier"] + "_" + results["variant"]
accuracy = results.pivot(index = ["dataset", "resample"], columns = "run", values = "accuracy")
accuracy = accuracy[[f"{classifier}_{variant}" for classifier in classifiers for variant in variants]]

reproduction = pd.read_csv(REPRODUCTION_CSV).set_index(["dataset", "resample"])["accuracy"]
accuracy["reproduction"] = reproduction.reindex(accuracy.index)

summary = accuracy.groupby("dataset").mean()
for classifier in classifiers:
    for variant in others:
        summary[f"{classifier}_{variant}_minus_{BASELINE}"] = summary[f"{classifier}_{variant}"] - summary[f"{classifier}_{BASELINE}"]

summary["resamples"] = accuracy.groupby("dataset").size()

time_columns = ["fit_seconds", "transform_seconds", "train_seconds", "total_seconds"]
timing = results.pivot_table(index = "dataset", columns = "run", values = time_columns, aggfunc = "mean")
timing.columns = [f"{run}_{step}" for step, run in timing.columns]
summary = summary.join(timing)

f1 = results.pivot(index = ["dataset", "resample"], columns = "run", values = "f1").groupby("dataset").mean()
f1.columns = [f"{run}_f1" for run in f1.columns]
summary = summary.join(f1)

summary.to_csv(SUMMARY_CSV)

print(f"Datasets: {len(summary)} | resamples per dataset: {summary['resamples'].min()} to {summary['resamples'].max()}")
print(f"Mean accuracy over datasets | reproduction: {summary['reproduction'].mean():.4f}")

for classifier in classifiers:
    print(f"\n{classifier}")
    print(f"  {BASELINE:12} {summary[f'{classifier}_{BASELINE}'].mean():.4f}")
    print(f"  {'':12} F1 {summary[f'{classifier}_{BASELINE}_f1'].mean():.4f}")
    for variant in others:
        difference = summary[f"{classifier}_{variant}_minus_{BASELINE}"]
        wins = int((difference > 1e-9).sum())
        losses = int((difference < -1e-9).sum())
        draws = int((difference.abs() <= 1e-9).sum())
        print(f"  {variant:12} {summary[f'{classifier}_{variant}'].mean():.4f} | vs {BASELINE}: {difference.mean():+.4f} " f"| wins / draws / losses: {wins} / {draws} / {losses}")
        f1_difference = summary[f"{classifier}_{variant}_f1"] - summary[f"{classifier}_{BASELINE}_f1"]
        print(f"  {'':12} F1 {summary[f'{classifier}_{variant}_f1'].mean():.4f} | vs {BASELINE}: {f1_difference.mean():+.4f}")


print(f"\nSummary saved to {SUMMARY_CSV}")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
summary.round(4)

Datasets: 109 | resamples per dataset: 30 to 30
Mean accuracy over datasets | reproduction: 0.8724

ridge
  raw          0.8723
               F1 0.8490
  aa_sf_0_1    0.8763 | vs raw: +0.0040 | wins / draws / losses: 52 / 11 / 46
               F1 0.8530 | vs raw: +0.0041

Summary saved to ../results_improvements/aa_summary.csv


,ridge_raw,ridge_aa_sf_0_1,reproduction,ridge_aa_sf_0_1_minus_raw,resamples,ridge_aa_sf_0_1_fit_seconds,ridge_raw_fit_seconds,ridge_aa_sf_0_1_total_seconds,ridge_raw_total_seconds,ridge_aa_sf_0_1_train_seconds,ridge_raw_train_seconds,ridge_aa_sf_0_1_transform_seconds,ridge_raw_transform_seconds,ridge_aa_sf_0_1_f1,ridge_raw_f1
dataset,,,,,,,,,,,,,,,
ACSF1,0.8237,0.8350,0.8247,0.0113,30,0.2440,0.1669,0.5127,0.3833,0.0488,0.0495,0.2188,0.1659,0.8341,0.8218
Adiac,0.8084,0.8063,0.8086,-0.0021,30,0.0423,0.0337,0.3392,0.3107,0.1812,0.1830,0.1125,0.0925,0.7938,0.7962
ArrowHead,0.8800,0.8806,0.8813,0.0006,30,0.0673,0.0503,0.1317,0.1074,0.0206,0.0217,0.0427,0.0349,0.8805,0.8799
BME,0.9922,0.9927,0.9920,0.0004,30,0.0477,0.0386,0.0867,0.0743,0.0176,0.0174,0.0205,0.0179,0.9927,0.9922
Beef,0.7844,0.7722,0.7878,-0.0122,30,0.0980,0.0729,0.1386,0.1098,0.0184,0.0173,0.0218,0.0194,0.7653,0.7779
BeetleFly,0.9033,0.9133,0.9083,0.0100,30,0.1046,0.0765,0.1298,0.0994,0.0114,0.0115,0.0138,0.0114,0.9124,0.9024
BirdChicken,0.8967,0.9000,0.9017,0.0033,30,0.1017,0.0737,0.1266,0.0965,0.0114,0.0114,0.0135,0.0113,0.8990,0.8955
CBF,0.9971,0.9984,0.9971,0.0013,30,0.0516,0.0411,0.1859,0.1522,0.0284,0.0289,0.1015,0.0802,0.9984,0.9971
Car,0.9200,0.9128,0.9172,-0.0072,30,0.1045,0.0800,0.1772,0.1419,0.0249,0.0244,0.0471,0.0371,0.9156,0.9223
